# Material Balance Template

**A worked steady-state material balance, from a hand-checkable two-stream mixer to a
three-unit flowsheet with recycle.**

| | |
| --- | --- |
| **Purpose** | A starting point for material balance problems, structured so the linear-algebra machinery is separated from the problem definition |
| **Author** | Open ChemE Hub contributors |
| **Licence** | MIT |
| **Depends** | `numpy`, `matplotlib` (`pip install numpy matplotlib`) |

## What to change first

Sections 2 and 4 define the problems. Everything else is machinery that works unchanged for
any linear steady-state balance. Replace the stoichiometry, the split fractions, and the
feed specification with yours.

## How to use this

Work through it once with the numbers as given, checking Section 1 by hand — you should be
able to do that balance on paper in two minutes, and seeing it agree with the code is the
point of doing it first. Then replace the problem definition.

## Scope and assumptions

- **Steady state.** No accumulation term. For batch or startup you need `dM/dt`, which is a
  different (ODE) problem.
- **No reaction** in Sections 1–3; Section 4 adds a single reaction with known conversion.
- **All balances are linear** in the unknown flows, so they solve directly with `numpy.linalg.solve`.
  A balance with an unknown split *and* an unknown composition multiplying each other is
  nonlinear — see the note at the end of Section 4.
- Mass basis throughout unless stated. Mixing mass and mole bases in one balance is the most
  common source of a balance that won't close.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})

# Tolerance for calling a balance closed. 1e-6 relative is numerical noise;
# anything above ~1e-3 relative on real data means a stream is missing.
TOL = 1e-9

print(f"numpy {np.__version__}")

---

## 1. The simplest case: a two-stream mixer

Two streams combine. Nothing reacts, nothing accumulates.

```
  S1  (1000 kg/h, 40% ethanol / 60% water)
        \
         >---- MIXER ----> S3  (? kg/h, ?% ethanol)
        /
  S2  ( 500 kg/h, 10% ethanol / 90% water)
```

Two equations, two unknowns ($\dot m_3$ and $x_{3,\mathrm{EtOH}}$):

$$\text{overall:}\quad \dot m_1 + \dot m_2 = \dot m_3$$
$$\text{ethanol:}\quad \dot m_1 x_{1} + \dot m_2 x_{2} = \dot m_3 x_{3}$$

Do this one on paper first. If the code below disagrees with your arithmetic, trust the paper.

In [ ]:
# --- Problem definition -----------------------------------------------
m1, x1_etoh = 1000.0, 0.40   # kg/h, mass fraction ethanol
m2, x2_etoh = 500.0, 0.10

# --- Solution ---------------------------------------------------------
m3 = m1 + m2                                     # overall balance
x3_etoh = (m1 * x1_etoh + m2 * x2_etoh) / m3     # ethanol balance

print(f"Product stream: {m3:.1f} kg/h at {x3_etoh:.4f} mass fraction ethanol")
print(f"                = {m3 * x3_etoh:.1f} kg/h ethanol + {m3 * (1 - x3_etoh):.1f} kg/h water")

# --- Check: the balance must close on every component -----------------
etoh_in  = m1 * x1_etoh + m2 * x2_etoh
etoh_out = m3 * x3_etoh
water_in  = m1 * (1 - x1_etoh) + m2 * (1 - x2_etoh)
water_out = m3 * (1 - x3_etoh)

print(f"\nEthanol : in {etoh_in:8.2f}  out {etoh_out:8.2f}  closure {abs(etoh_in - etoh_out):.2e} kg/h")
print(f"Water   : in {water_in:8.2f}  out {water_out:8.2f}  closure {abs(water_in - water_out):.2e} kg/h")

assert abs(etoh_in - etoh_out) < TOL * max(etoh_in, 1.0), "Ethanol balance does not close"
assert abs(water_in - water_out) < TOL * max(water_in, 1.0), "Water balance does not close"
print("\nBoth components close. This is the check to run after every unit, every time.")

---

## 2. The general form: balances as a linear system

Hand algebra stops scaling at about four streams. The general statement of a steady-state
linear balance is

$$\mathbf{A}\,\mathbf{x} = \mathbf{b}$$

where each **row** is one conservation equation (total, or one component) and each **column**
is one unknown stream flow. Building `A` explicitly is worth the extra typing: you can read
the physics off the matrix, and a singular `A` tells you immediately that the problem is
under-specified rather than leaving you to hunt for the missing spec.

### The flowsheet

A dilute ethanol feed is concentrated in a distillation column; the bottoms go to a
settler that returns some water to the feed.

```
        F                         D  (distillate, 85% EtOH)
   ---------> [ MIXER ] --M--> [ COLUMN ]
                  ^                 |
                  |                 v  B  (bottoms, 2% EtOH)
                  |            [ SETTLER ]
                  |                 |
                  +------ R --------+---> W  (waste water)
              (recycle)
```

### Specification

| Quantity | Value |
| --- | --- |
| Feed `F` | 10 000 kg/h, 12% ethanol by mass |
| Distillate composition | 85% ethanol |
| Bottoms composition | 2% ethanol |
| Settler recycle | 30% of the bottoms stream is recycled |

### Degrees of freedom

Unknowns: `M`, `D`, `B`, `R`, `W`, and the mixed-stream composition `x_M` — six.
Independent equations: two components on each of the mixer, column, and settler — six.
Zero degrees of freedom, so it solves.

`x_M` multiplies `M`, so we solve for the *component flows* $\dot m_i x_i$ rather than the
composition. That keeps the system linear — a standard trick, and the reason the vector below
holds ethanol and water flows separately.

In [ ]:
# --- Problem definition -----------------------------------------------
F        = 10_000.0   # kg/h feed
xF       = 0.12       # mass fraction ethanol in feed
xD       = 0.85       # mass fraction ethanol in distillate (spec)
xB       = 0.02       # mass fraction ethanol in bottoms (spec)
recycle_fraction = 0.30   # fraction of bottoms returned to the mixer

# --- Unknowns, in the order the columns of A appear --------------------
#   0: M  mixed feed to column      3: R  recycle to mixer
#   1: D  distillate                4: W  waste water
#   2: B  column bottoms
names = ["M", "D", "B", "R", "W"]
n = len(names)

A = np.zeros((n, n))
b = np.zeros(n)

# Row 0 -- MIXER, total mass:      F + R = M
A[0, names.index("M")] = -1.0
A[0, names.index("R")] = +1.0
b[0] = -F

# Row 1 -- COLUMN, total mass:     M = D + B
A[1, names.index("M")] = +1.0
A[1, names.index("D")] = -1.0
A[1, names.index("B")] = -1.0
b[1] = 0.0

# Row 2 -- COLUMN, ethanol:        M*xM = D*xD + B*xB
# xM is unknown, but M*xM = F*xF + R*xB  (recycle has bottoms composition),
# so substitute and the equation stays linear in M, D, B, R:
#        F*xF + R*xB = D*xD + B*xB
A[2, names.index("R")] = +xB
A[2, names.index("D")] = -xD
A[2, names.index("B")] = -xB
b[2] = -F * xF

# Row 3 -- SETTLER, total mass:    B = R + W
A[3, names.index("B")] = +1.0
A[3, names.index("R")] = -1.0
A[3, names.index("W")] = -1.0
b[3] = 0.0

# Row 4 -- SETTLER split spec:     R = recycle_fraction * B
A[4, names.index("R")] = +1.0
A[4, names.index("B")] = -recycle_fraction
b[4] = 0.0

print("Coefficient matrix A:")
print("        " + "".join(f"{nm:>10s}" for nm in names))
for i, row in enumerate(A):
    print(f"  eq{i}: " + "".join(f"{v:>10.3f}" for v in row) + f"   | {b[i]:>10.1f}")

cond = np.linalg.cond(A)
print(f"\nCondition number: {cond:.3g}")
if cond > 1e10:
    print("  WARNING: nearly singular -- the problem is under-specified or a spec is redundant.")

In [ ]:
# --- Solve -------------------------------------------------------------
x = np.linalg.solve(A, b)
flows = dict(zip(names, x))
flows["F"] = F

print("Stream table")
print("=" * 58)
print(f"  {'Stream':<10s}{'kg/h':>12s}{'x_EtOH':>10s}{'EtOH kg/h':>12s}{'H2O kg/h':>12s}")
print("  " + "-" * 54)

# Compositions: known by specification for every stream except the mixed feed.
xM = (F * xF + flows["R"] * xB) / flows["M"]
comps = {"F": xF, "M": xM, "D": xD, "B": xB, "R": xB, "W": xB}

for s in ["F", "M", "D", "B", "R", "W"]:
    m, xs = flows[s], comps[s]
    print(f"  {s:<10s}{m:>12,.1f}{xs:>10.4f}{m * xs:>12,.1f}{m * (1 - xs):>12,.1f}")

print(f"\nMixed feed composition x_M = {xM:.4f} "
      f"(diluted from {xF:.4f} by the recycle -- recycle always dilutes here,")
print("because the returned stream is leaner in ethanol than the fresh feed)")

### Closing the balance

A stream table that hasn't been checked is a hypothesis. Two independent checks:

1. **Overall envelope** — draw a box around the entire flowsheet. Only `F` enters; only `D`
   and `W` leave. The recycle is internal and must not appear.
2. **Unit by unit** — every component, every unit. This is what localises an error when the
   overall check fails.

In [ ]:
def check_balance(label, streams_in, streams_out, flows, comps, tol=1e-6):
    '''Check total and ethanol balances around one envelope.

    streams_in / streams_out : lists of stream names.
    Returns True if both close to within `tol` relative.
    '''
    m_in  = sum(flows[s] for s in streams_in)
    m_out = sum(flows[s] for s in streams_out)
    e_in  = sum(flows[s] * comps[s] for s in streams_in)
    e_out = sum(flows[s] * comps[s] for s in streams_out)

    rel_m = abs(m_in - m_out) / max(m_in, 1.0)
    rel_e = abs(e_in - e_out) / max(e_in, 1.0)
    ok = rel_m < tol and rel_e < tol

    print(f"  {label:<22s} total {m_in:>10,.1f} -> {m_out:>10,.1f}  (rel {rel_m:.2e})")
    print(f"  {'':<22s} EtOH  {e_in:>10,.1f} -> {e_out:>10,.1f}  (rel {rel_e:.2e})   "
          f"{'OK' if ok else 'FAILS'}")
    return ok


print("Balance checks")
print("=" * 72)
results = [
    check_balance("Overall envelope",  ["F"],       ["D", "W"], flows, comps),
    check_balance("Mixer",             ["F", "R"],  ["M"],      flows, comps),
    check_balance("Column",            ["M"],       ["D", "B"], flows, comps),
    check_balance("Settler",           ["B"],       ["R", "W"], flows, comps),
]

print()
assert all(results), "At least one balance failed -- fix the flowsheet before going further"
print("All balances close. The stream table is arithmetically consistent.")
print("\nNote what this does NOT tell you: whether the column can actually make an 85%")
print("distillate. Ethanol-water azeotropes at about 95.6 wt% at 1 atm, so 85% is")
print("achievable -- but a material balance would happily have returned 98% too.")
print("Feasibility is a thermodynamics question, and it is a separate one.")

---

## 3. Visualising the balance

Two views worth generating every time:

- **Component flows per stream** — makes a misplaced order of magnitude obvious at a glance.
- **Recycle sensitivity** — how the internal flows respond to the recycle ratio. This is the
  plot that justifies (or kills) a recycle: internal flows grow much faster than the ratio does.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left: component flows by stream ----------------------------------
order = ["F", "M", "D", "B", "R", "W"]
etoh  = np.array([flows[s] * comps[s] for s in order])
water = np.array([flows[s] * (1 - comps[s]) for s in order])
idx = np.arange(len(order))

ax1.bar(idx, etoh, label="Ethanol", color="#3b7dd8")
ax1.bar(idx, water, bottom=etoh, label="Water", color="#a8c8f0")
ax1.set_xticks(idx, order)
ax1.set_ylabel("Mass flow [kg/h]")
ax1.set_title("Component flows by stream")
ax1.legend()

for i, s in enumerate(order):
    ax1.text(i, flows[s] * 1.02, f"{flows[s]:,.0f}", ha="center", fontsize=8)

# --- Right: sensitivity to recycle fraction ---------------------------
fractions = np.linspace(0.0, 0.85, 40)
M_vals, R_vals, W_vals = [], [], []

for rf in fractions:
    A2 = A.copy()
    A2[4, names.index("B")] = -rf
    sol = dict(zip(names, np.linalg.solve(A2, b)))
    M_vals.append(sol["M"])
    R_vals.append(sol["R"])
    W_vals.append(sol["W"])

ax2.plot(fractions, M_vals, lw=2, label="M (column feed)")
ax2.plot(fractions, R_vals, lw=2, label="R (recycle)")
ax2.plot(fractions, W_vals, lw=2, label="W (waste)")
ax2.axvline(recycle_fraction, color="crimson", ls="--", lw=1.2,
            label=f"design point ({recycle_fraction:.0%})")
ax2.set_xlabel("Recycle fraction of bottoms [-]")
ax2.set_ylabel("Mass flow [kg/h]")
ax2.set_title("Sensitivity to recycle ratio")
ax2.legend(fontsize=8)

fig.tight_layout()
plt.show()

print(f"At {recycle_fraction:.0%} recycle the column sees {flows['M']:,.0f} kg/h, "
      f"{flows['M'] / F - 1:.1%} more than the fresh feed.")
print("The column is sized for M, not F. That is what recycle costs you in capital,")
print("and why the curve steepening past ~60% is where recycles stop paying.")

---

## 4. Adding a reaction

With reaction, the total mass balance still holds (mass is conserved) but individual
component balances gain a generation term:

$$\text{in} - \text{out} + \text{generation} = 0$$

Worked case: **methanol synthesis**, $\mathrm{CO} + 2\,\mathrm{H_2} \rightarrow \mathrm{CH_3OH}$,
single pass at known conversion. Molar basis, because stoichiometry is a molar statement —
then converted back to mass to check conservation.

Specification: 100 kmol/h CO and 250 kmol/h H₂ fed, 65% conversion of CO.

In [ ]:
# --- Problem definition -----------------------------------------------
species      = ["CO", "H2", "CH3OH"]
nu           = np.array([-1.0, -2.0, +1.0])       # stoichiometric coefficients
MW           = np.array([28.010, 2.016, 32.042])  # kg/kmol
n_in         = np.array([100.0, 250.0, 0.0])      # kmol/h
conversion   = 0.65                                # of the limiting reactant

# --- Which reactant limits? -------------------------------------------
# Compare feed moles to stoichiometric requirement; the smallest ratio limits.
reactant_idx = [i for i, v in enumerate(nu) if v < 0]
ratios = {species[i]: n_in[i] / abs(nu[i]) for i in reactant_idx}
limiting = min(ratios, key=ratios.get)
i_lim = species.index(limiting)

print("Feed-to-stoichiometry ratios:")
for s, r in ratios.items():
    print(f"  {s:<8s}{r:>10.2f} kmol/h per unit stoichiometry"
          f"{'   <-- limiting' if s == limiting else ''}")

# --- Extent of reaction -----------------------------------------------
extent = conversion * n_in[i_lim] / abs(nu[i_lim])     # kmol/h
n_out  = n_in + nu * extent

print(f"\nExtent of reaction: {extent:.3f} kmol/h at {conversion:.0%} conversion of {limiting}")
print("\nMolar balance")
print("=" * 62)
print(f"  {'Species':<10s}{'in [kmol/h]':>14s}{'out [kmol/h]':>14s}{'change':>12s}")
for i, s in enumerate(species):
    print(f"  {s:<10s}{n_in[i]:>14.3f}{n_out[i]:>14.3f}{n_out[i] - n_in[i]:>12.3f}")

if np.any(n_out < -1e-9):
    raise ValueError("Negative outlet moles -- conversion exceeds the stoichiometric limit")

# --- Mass check: moles are not conserved, mass always is ---------------
m_in, m_out = n_in * MW, n_out * MW
print("\nMass balance")
print("=" * 62)
print(f"  {'Species':<10s}{'in [kg/h]':>14s}{'out [kg/h]':>14s}")
for i, s in enumerate(species):
    print(f"  {s:<10s}{m_in[i]:>14.2f}{m_out[i]:>14.2f}")
print("  " + "-" * 58)
print(f"  {'TOTAL':<10s}{m_in.sum():>14.2f}{m_out.sum():>14.2f}")

rel = abs(m_in.sum() - m_out.sum()) / m_in.sum()
print(f"\n  Total moles:  {n_in.sum():.2f} -> {n_out.sum():.2f} kmol/h  (NOT conserved -- 3 moles make 1)")
print(f"  Total mass:   {m_in.sum():.2f} -> {m_out.sum():.2f} kg/h  (conserved, rel error {rel:.2e})")
assert rel < 1e-12, "Mass is not conserved -- check MW or stoichiometry"

# --- Product composition ----------------------------------------------
y_out = n_out / n_out.sum()
print("\nOutlet mole fractions:")
for s, y in zip(species, y_out):
    print(f"  y_{s:<8s} = {y:.4f}")

### When the balance stops being linear

Everything above solves in one `np.linalg.solve` call because every equation is linear in the
unknowns. Three common situations break that, and each needs a different tool:

| Situation | Why it's nonlinear | What to use |
| --- | --- | --- |
| Unknown flow **and** unknown composition in the same stream | Their product appears | Solve for component flows (as in §2), or `scipy.optimize.fsolve` |
| Recycle with a **conversion specified on the fresh feed** rather than per pass | Recycle flow depends on the result | Sequential substitution (tear the recycle, iterate to convergence) or a simultaneous solve |
| Equilibrium-limited reaction | $K_{eq}$ is nonlinear in composition | `fsolve` on the combined balance + equilibrium equations, or a proper flash routine |

For a flowsheet of any size, stop hand-rolling and use a simulator — DWSIM for a GUI,
[IDAES](https://github.com/IDAES/idaes-pse) if you want equation-oriented Python. The value of
doing it by hand is that you learn what the simulator is doing and can tell when it's wrong.

---

## Checklist before you trust a balance

- [ ] Every stream has a **basis** and **units** written next to it
- [ ] Mole and mass bases are not mixed within one equation
- [ ] Degrees of freedom counted: unknowns = independent equations
- [ ] Overall envelope closes (recycle streams absent from it)
- [ ] Every unit closes on every component
- [ ] No negative flows or compositions outside [0, 1]
- [ ] Compositions in each stream sum to 1
- [ ] The answer is **physically feasible** — not above an azeotrope, not past equilibrium,
      not requiring a separation that thermodynamics forbids
- [ ] Order of magnitude sanity: does the flow fit in a pipe a human would specify?

## Next steps

- Add an **energy balance** — see [`awesome-chemical-engineering`](https://github.com/open-cheme-hub/awesome-chemical-engineering) for property libraries (`thermo`, CoolProp, Cantera)
- Size the reactor with [`reactor_design_skeleton.py`](reactor_design_skeleton.py)
- Take it to a flowsheet simulator and compare — the disagreements are where you learn something